# Day 12 · Exercise 2: build_chroma_collection

**What you'll build:** `build_collection(client: chromadb.Client, name: str, docs: list[dict]) -> chromadb.Collection` — a function that creates (or reopens) a named Chroma collection, embeds each document's text with `nomic-embed-text`, and adds all documents with sequential IDs and metadata in a single `collection.add()` call.

**Why it matters:** This is the core write path of every vector-database-backed app — get this pattern right once and you will reuse it in semantic search, RAG pipelines, and recommendation systems for the rest of the course.

## Your Implementation

In [ ]:
import chromadb
import ollama

EMBED_MODEL = "nomic-embed-text"


def _embed(text: str) -> list[float]:
    """Return the embedding vector for a piece of text using nomic-embed-text."""
    response = ollama.embeddings(model=EMBED_MODEL, prompt=text)
    return response["embedding"]


def build_collection(
    client: chromadb.Client,
    name: str,
    docs: list[dict],
) -> chromadb.Collection:
    """
    Create (or reopen) a named Chroma collection and populate it with documents.

    Each item in `docs` must have at least a ``"text"`` key. Any additional
    keys are stored as metadata alongside the vector.

    The function embeds every document's text with ``nomic-embed-text`` and
    assigns sequential IDs in the form ``"doc_0"``, ``"doc_1"``, etc.

    Args:
        client: An open ``chromadb.Client`` (e.g. a ``PersistentClient``).
        name:   The name of the collection to create or reopen.
        docs:   A list of dicts, each with at minimum a ``"text"`` key.
                Example: ``[{"text": "Chroma stores vectors on disk.",
                             "source": "docs"}]``

    Returns:
        The populated ``chromadb.Collection`` object.

    Example:
        >>> import chromadb
        >>> client = chromadb.EphemeralClient()
        >>> sample_docs = [
        ...     {"text": "Vectors encode meaning as numbers."},
        ...     {"text": "Chroma persists embeddings to disk."},
        ... ]
        >>> col = build_collection(client, "demo", sample_docs)
        >>> col.count()
        2
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'


def _run_checks():
    score, total = 0, 4

    # Check 1: function is defined and callable
    try:
        assert callable(build_collection), 'build_collection is not defined'
        print(f'{_PASS} Check 1/{total}: build_collection is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return  # cannot continue — later checks would NameError

    # Build a shared in-memory client + two sample docs for checks 2-4
    _client = chromadb.EphemeralClient()
    _docs = [
        {"text": "Chroma stores vectors on disk for fast retrieval.", "topic": "storage"},
        {"text": "Cosine similarity measures the angle between two vectors.", "topic": "math"},
        {"text": "Metadata filters narrow results before vector search.", "topic": "filtering"},
    ]

    _col = None
    try:
        _col = build_collection(_client, "test_collection", _docs)
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: build_collection raised an error — {e}')
        print(f'{_FAIL} Check 3/{total}: skipped (build_collection failed)')
        print(f'{_FAIL} Check 4/{total}: skipped (build_collection failed)')
        print(f'\n  {score}/{total} passed. Keep going!')
        return

    # Check 2: returns a chromadb.Collection
    try:
        assert isinstance(_col, chromadb.Collection), (
            f'expected chromadb.Collection, got {type(_col).__name__}'
        )
        print(f'{_PASS} Check 2/{total}: returns a chromadb.Collection')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')
        return  # cannot query a non-Collection object safely

    # Check 3: collection contains the correct number of documents
    try:
        count = _col.count()
        assert count == len(_docs), (
            f'expected {len(_docs)} documents in collection, found {count}'
        )
        print(f'{_PASS} Check 3/{total}: collection.count() == {len(_docs)} (all docs added)')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: IDs follow the "doc_N" convention and query returns a result
    try:
        _items = _col.get()
        _ids = set(_items["ids"])
        _expected_ids = {f"doc_{i}" for i in range(len(_docs))}
        assert _ids == _expected_ids, (
            f'expected IDs {sorted(_expected_ids)}, got {sorted(_ids)}'
        )
        # Also verify querying works end-to-end
        from chromadb.utils import embedding_functions  # noqa: F401 — just import check
        _qvec = _embed("how are vectors stored?")
        _results = _col.query(query_embeddings=[_qvec], n_results=1)
        assert len(_results["ids"][0]) == 1, 'query returned no results'
        print(f'{_PASS} Check 4/{total}: IDs are "doc_0"…"doc_{len(_docs)-1}" and query returns results')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
    print(f'  {score}/{total} passed.' + ('' if score == total else ' Keep going!'))


_run_checks()

## Bonus Challenge

In Lesson 3 you will learn how to keep a live collection fresh with `upsert`. Get a head start: modify `build_collection` (or write a new `upsert_collection`) so it uses `collection.upsert(...)` instead of `collection.add(...)`. Then call it twice with the same docs and confirm `collection.count()` does **not** double — Chroma should detect duplicate IDs and overwrite rather than create new entries.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import chromadb
import ollama

EMBED_MODEL = "nomic-embed-text"


def _embed(text: str) -> list[float]:
    response = ollama.embeddings(model=EMBED_MODEL, prompt=text)
    return response["embedding"]


def build_collection(
    client: chromadb.Client,
    name: str,
    docs: list[dict],
) -> chromadb.Collection:
    collection = client.get_or_create_collection(name=name)

    ids = [f"doc_{i}" for i in range(len(docs))]
    texts = [d["text"] for d in docs]
    embeddings = [_embed(t) for t in texts]
    metadatas = [{k: v for k, v in d.items() if k != "text"} for d in docs]

    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=texts,
        metadatas=metadatas,
    )
    return collection
```

**Why this works:** `get_or_create_collection` is idempotent — it creates the collection on the first run and silently reopens it on every subsequent run, so your setup code is safe to call repeatedly. The four parallel lists passed to `collection.add()` — `ids`, `embeddings`, `documents`, and `metadatas` — must be the same length and in the same order; Chroma writes them atomically to disk so the data survives process restarts. Stripping the `"text"` key from each doc before building `metadatas` keeps the metadata dict clean, since Chroma already stores the raw text in the `documents` field.
</details>